## Learning notes — 03. Process Addresses Data

### Learning objectives
Follow **03. Process Addresses Data** from its source inputs through its operations to the archived results.
- Explain stream, autoloader, scd, profiling as used in this lesson.

### Source-specific learning focus
- Follow incremental ingestion from discovery to persisted results.
- Separate file discovery, parsing and schema tracking.
- Trace sequenced changes into current-state or history-preserving records.
- Interpret selected fields, samples and counts at their actual output grain.

### Input/output interpretation
Original source and outputs are preserved unchanged and were not executed. Saved tables, HTML displays and errors are archived generated outputs, not supplied datasets. Inline literals are examples already present in the source. Cloud storage, remote tables, catalogs, services and checkpoints are unavailable locally unless a supplied snapshot is explicitly listed below; no external input is claimed available.


## Learning notes — supplied input snapshot

Path: `data/circuitbox/addresses/addresses_day1.csv`

SHA-256: `52ef2bd805cb67c39f04fa83c36b3820627e709904520858f1d79fd54671a327`

Complete supplied text (not generated output):

```csv
customer_id,address_line_1,city,state,postcode,created_date
1001,9713 Medina Mountains Apt. 813,Lake Erika,Mississippi,91945,2024-10-01
1002,084 Anne Hollow Apt. 064,East Jasontown,Minnesota,77329,2024-10-01
1003,8400 Rebecca Lodge Suite 011,Lake Lisatown,Montana,11354,2024-10-01
1004,869 Linda Locks Apt. 105,West Melissabury,Michigan,3766,2024-10-01
1005,4838 Jacob Neck,Jamesview,Wisconsin,58657,2024-10-01
1006,4558 Cabrera Islands,Lake David,,06133,2024-10-01
1007,75470 Richardson Mission Apt. 147,New Rachelhaven,Hawaii,38857,2024-10-01
1008,4450 Michelle Mission Suite 663,North Justin,Kentucky,91757,2024-10-01
1009,331 Monica Route Apt. 022,Michellechester,Texas,14637,2024-10-01
1010,48794 Aguirre Ferry Apt. 952,North Amy,Washington,92507,2024-10-01
```

## Learning notes — supplied input snapshot

Path: `data/circuitbox/addresses/addresses_day2.csv`

SHA-256: `b39f1e2d89f68564db4df92102118c8270a1ac7190494c7dc5488f2b3c3fd8c6`

Complete supplied text (not generated output):

```csv
customer_id,address_line_1,city,state,postcode,created_date
1001,13677 Garcia Ports,Shelleyland,Rhode Island,03719,2025-01-14
1004,955 Montgomery Mountain Suite 167,Jameschester,Kentucky,09013,2025-01-14
1006,5094 Ayala Falls,Greenmouth,North Carolina,02638,2025-01-14
1011,2856 Jeremy Wall Apt. 985,East Elizabeth,Oklahoma,66460,2025-01-14
1012,14540 Moody Tunnel,West Shellybury,,83151,2025-01-14
1013,5733 Richards Ferry Suite 995,East Angelafort,Indiana,4623,2025-01-14
```

# Archived Databricks example

Historical source and outputs, not locally executable. Nothing was run during conversion. Databricks services, datasets and original image assets may be unavailable; image references are preserved, not downloaded. Outputs and errors are archived, not new execution.

## Process Addresses Data 
1. Ingest the data into the data lakehouse - bronze_addresses
1. Perform data quality checks and transform the data as required - silver_addresses_clean
1. Apply changes to the Addresses data (SCD Type 2) - silver_addresses

In [ ]:
import dlt
import pyspark.sql.functions as F

### 1. Ingest the data into the data lakehouse - bronze_addresses

![dea06-03-process-address-data-1.png](images/dea06-03-process-address-data-1.png)

## Learning notes — before source operation

**stream**: A streaming input is processed incrementally, not as a fixed batch. Interpret displayed rows as historical observations, not proof that a stream is currently running.

**autoloader**: Auto Loader discovers arriving files. File format/options and schema tracking determine parsing; discovery is distinct from downstream transformations.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
@dlt.table(
    name = "bronze_addresses",
    table_properties = {'quality' : 'bronze'},
    comment = "Raw addresses data ingested from the source system"
)
def create_bronze_addresses():
    return (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.inferColumnTypes", "true")
            .load("/Volumes/circuitbox/landing/operational_data/addresses/")
            .select(
                "*",
                F.col("_metadata.file_path").alias("input_file_path"),
                F.current_timestamp().alias("ingest_timestamp")
            )
    )

### 2. Perform data quality checks and transform the data as required - silver_addresses_clean

## Learning notes — before source operation

**stream**: A streaming input is processed incrementally, not as a fixed batch. Interpret displayed rows as historical observations, not proof that a stream is currently running.

**profiling**: Read the selected columns and output grain before interpreting the saved result. Counts, nulls and samples describe the archived input state, not a newly executed validation.

In [ ]:
@dlt.table(
    name = "silver_addresses_clean",
    comment = "Cleaned addresses data",
    table_properties = {'quality' : 'silver'}
)
@dlt.expect_or_fail("valid_customer_id", "customer_id IS NOT NULL")
@dlt.expect_or_drop("valid_address", "address_line_1 IS NOT NULL")
@dlt.expect("valid_postcode", "LENGTH(postcode) = 5")
def create_silver_addresses_clean():
    return (
        spark.readStream.table("LIVE.bronze_addresses")
            .select(
                "customer_id",
                "address_line_1",
                "city",
                "state",
                "postcode",
                F.col("created_date").cast("date")
            )
    )

### 3. Apply changes to the Addresses data (SCD Type 2) - silver_addresses

![dea06-03-process-address-data-2.png](images/dea06-03-process-address-data-2.png)

## Learning notes — before source operation

**stream**: A streaming input is processed incrementally, not as a fixed batch. Interpret displayed rows as historical observations, not proof that a stream is currently running.

**scd**: Change application uses keys and sequencing to resolve events. SCD type 1 replaces current values; type 2 retains validity intervals. Verify the actual configured type and event ordering.

**Unexecuted teaching example (illustrative, not an archived or generated result):** For a customer address change, SCD1 replaces the address; SCD2 can keep the old and new address in separate validity intervals.

In [ ]:
dlt.create_streaming_table(
    name = "silver_addresses",
    comment = "SCD Type 2 addresses data",
    table_properties = {'quality' : 'silver'}
)

## Learning notes — before source operation

**scd**: Change application uses keys and sequencing to resolve events. SCD type 1 replaces current values; type 2 retains validity intervals. Verify the actual configured type and event ordering.

**Unexecuted teaching example (illustrative, not an archived or generated result):** For a customer address change, SCD1 replaces the address; SCD2 can keep the old and new address in separate validity intervals.

In [ ]:
dlt.apply_changes(
    target = "silver_addresses",
    source = "silver_addresses_clean",
    keys = ["customer_id"],
    sequence_by = "created_date",
    stored_as_scd_type = 2
)